# 🚗 Phone-Robot: End-to-End 自動運転モデル学習 & TFLite量子化パイプライン

このノートブックは、トイラジコン（ジョーゼン 1/16 トヨタ86）を **Phone-as-a-Brain（スマホが頭脳）** で完全自律走行させるための、**End-to-End CNNモデルの学習およびTensorFlow Lite（TFLite）量子化エクスポート** を自動実行する完全パイプラインです。

### 📋 パイプライン概要
1. **GPU環境チェック**: GPU加速（T4 / V100 / A100）の確認
2. **公開走行データセットの取得**: Udacity / Donkey Car 互換の走行ログと画像データを自動クローン
3. **前処理パイプライン（事前仕様固定）**:
   - 天井・ボンネットをカットするROIクロップ
   - スマホ推論に最適な `(160, 120)` 解像度へリサイズ
   - 左右反転（反転時ステアリング符号反転）や明暗データ拡張
4. **モデル構築**: スマホ（Galaxy Note 10+）でリアルタイム動作する軽量CNN（マルチタスク: Steering & Throttle）
5. **GPU学習**: EarlyStopping & コサイン減衰を用いた最適化
6. **ステアリングのオンオフ評価**: 連続値出力を `[-1 (左), 0 (直進), 1 (右)]` のしきい値判定にマップする評価
7. **TFLite量子化エクスポート**:
   - `model_e2e_fp16.tflite`（モバイルGPU向け）
   - `model_e2e_int8.tflite`（モバイルNPU/CPU向け完全整数量子化）
8. **スマホ実機シミュレーション**: 推論レイテンシ（ms）と推定FPSの計測


In [ ]:
# 1. GPU環境の確認
!nvidia-smi

import os
import sys
import glob
import time
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import cv2

import tensorflow as tf
from tensorflow.keras import layers, models, callbacks

print(f"TensorFlow Version: {tf.__version__}")
print(f"GPU Available: {tf.config.list_physical_devices('GPU')}")


In [ ]:
# 2. 公開走行データセットの自動取得
DATA_DIR = "data"

if not os.path.exists(DATA_DIR):
    print("データセットをGitHubからクローンしています...")
    !git clone --depth 1 https://github.com/sgrpanchal31/self-driving-car-dataset.git data
    print("クローン完了!")
else:
    print("データセットは既に存在します。")

# driving_log.csv の確認
log_path = os.path.join(DATA_DIR, "driving_log.csv")
columns = ['center', 'left', 'right', 'steering', 'throttle', 'reverse', 'speed']
df = pd.read_csv(log_path, names=columns)
print(f"総フレーム数: {len(df)}")
df.head()


In [ ]:
# 3. 操舵角（Steering）の分布確認と前処理
# パスをColab上の相対パスに補正
def fix_path(p):
    filename = os.path.basename(p.strip())
    return os.path.join(DATA_DIR, "IMG", filename)

df['center_path'] = df['center'].apply(fix_path)
df['left_path'] = df['left'].apply(fix_path)
df['right_path'] = df['right'].apply(fix_path)

# 画像が存在するもののみ抽出
valid_df = df[df['center_path'].apply(os.path.exists)].reset_index(drop=True)
print(f"有効なセンター画像数: {len(valid_df)}")

plt.figure(figsize=(10, 4))
plt.hist(valid_df['steering'], bins=31, color='skyblue', edgecolor='black')
plt.title("Steering Angle Distribution (Raw)")
plt.xlabel("Steering Angle (-1.0 to 1.0)")
plt.ylabel("Frame Count")
plt.grid(True, alpha=0.3)
plt.show()


In [ ]:
# 4. 入力画像の前処理関数（仕様固定）
TARGET_WIDTH = 160
TARGET_HEIGHT = 120

def preprocess_image(img_bgr):
    # 1. BGRからRGBへ変換
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    
    # 2. ROIクロップ（天井40%と下部車体10%をカット）
    # 元画像サイズ: 160x320
    h, w, _ = img_rgb.shape
    crop_top = int(h * 0.40)
    crop_bottom = int(h * 0.90)
    cropped = img_rgb[crop_top:crop_bottom, 0:w]
    
    # 3. 目標解像度 (160x120) にリサイズ
    resized = cv2.resize(cropped, (TARGET_WIDTH, TARGET_HEIGHT))
    
    # 4. 正規化 (0.0 〜 1.0)
    normalized = resized.astype(np.float32) / 255.0
    return normalized

# サンプル画像の表示
sample_path = valid_df['center_path'].iloc[100]
raw_img = cv2.imread(sample_path)
processed_img = preprocess_image(raw_img)

plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1)
plt.imshow(cv2.cvtColor(raw_img, cv2.COLOR_BGR2RGB))
plt.title(f"Original Raw ({raw_img.shape[1]}x{raw_img.shape[0]})")
plt.axis('off')

plt.subplot(1, 2, 2)
plt.imshow(processed_img)
plt.title(f"ROI Cropped & Resized ({TARGET_WIDTH}x{TARGET_HEIGHT})")
plt.axis('off')
plt.tight_layout()
plt.show()


In [ ]:
# 5. マルチカメラ＆データ拡張ジェネレータ
def build_dataset_arrays(df_data, max_samples=4000):
    images = []
    steerings = []
    throttles = []
    
    # 左右カメラの操舵補正値
    STEERING_CORRECTION = 0.20
    
    for idx, row in df_data.iterrows():
        if len(images) >= max_samples:
            break
            
        c_path = row['center_path']
        st = float(row['steering'])
        th = float(row['throttle']) if 'throttle' in row else 0.5
        
        # 1. センターカメラ
        if os.path.exists(c_path):
            img = cv2.imread(c_path)
            if img is not None:
                p_img = preprocess_image(img)
                images.append(p_img)
                steerings.append(st)
                throttles.append(th)
                
                # データ拡張: 左右反転
                if abs(st) > 0.05:
                    images.append(cv2.flip(p_img, 1))
                    steerings.append(-st)
                    throttles.append(th)
                    
        # 2. 左右カメラによるリカバリーデータの追加
        l_path = row['left_path']
        if os.path.exists(l_path) and len(images) < max_samples:
            l_img = cv2.imread(l_path)
            if l_img is not None:
                images.append(preprocess_image(l_img))
                steerings.append(st + STEERING_CORRECTION)
                throttles.append(th)
                
    X = np.array(images, dtype=np.float32)
    y_steer = np.array(steerings, dtype=np.float32)
    y_throttle = np.array(throttles, dtype=np.float32)
    return X, y_steer, y_throttle

print("データセット配列を構築中...")
X, y_steer, y_throttle = build_dataset_arrays(valid_df, max_samples=4500)
print(f"構築完了: X={X.shape}, y_steer={y_steer.shape}, y_throttle={y_throttle.shape}")

# 学習用 / 検証用の分割 (80:20)
from sklearn.model_selection import train_test_split
X_train, X_val, y_s_train, y_s_val, y_t_train, y_t_val = train_test_split(
    X, y_steer, y_throttle, test_size=0.20, random_state=42
)
print(f"Train samples: {len(X_train)}, Val samples: {len(X_val)}")


In [ ]:
# 6. スマホ（Galaxy Note 10+）向け軽量CNNモデルの構築
def create_phone_robot_model(input_shape=(120, 160, 3)):
    inputs = layers.Input(shape=input_shape, name="camera_input")
    
    # 畳み込み特徴抽出
    x = layers.Conv2D(16, (5, 5), strides=(2, 2), padding='same', activation='relu')(inputs)
    x = layers.BatchNormalization()(x)
    
    x = layers.Conv2D(32, (5, 5), strides=(2, 2), padding='same', activation='relu')(x)
    x = layers.BatchNormalization()(x)
    
    x = layers.Conv2D(48, (3, 3), strides=(2, 2), padding='same', activation='relu')(x)
    x = layers.BatchNormalization()(x)
    
    x = layers.Conv2D(64, (3, 3), strides=(1, 1), padding='same', activation='relu')(x)
    x = layers.BatchNormalization()(x)
    
    x = layers.Flatten()(x)
    x = layers.Dense(64, activation='relu')(x)
    x = layers.Dropout(0.2)(x)
    x = layers.Dense(32, activation='relu')(x)
    
    # 出力1: Steering (-1.0 to 1.0)
    out_steering = layers.Dense(1, activation='tanh', name='steering_output')(x)
    
    # 出力2: Throttle (0.0 to 1.0)
    out_throttle = layers.Dense(1, activation='sigmoid', name='throttle_output')(x)
    
    model = models.Model(inputs=inputs, outputs=[out_steering, out_throttle], name="PhoneRobot_E2E")
    return model

model = create_phone_robot_model()
model.summary()

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss={'steering_output': 'mse', 'throttle_output': 'mse'},
    loss_weights={'steering_output': 1.0, 'throttle_output': 0.2}
)


In [ ]:
# 7. GPUを使ったモデル学習
callbacks_list = [
    callbacks.EarlyStopping(monitor='val_loss', patience=4, restore_best_weights=True),
    callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=2, min_lr=1e-5)
]

print("学習を開始します（GPU加速中）...")
start_time = time.time()
history = model.fit(
    X_train, {'steering_output': y_s_train, 'throttle_output': y_t_train},
    validation_data=(X_val, {'steering_output': y_s_val, 'throttle_output': y_t_val}),
    epochs=15,
    batch_size=64,
    callbacks=callbacks_list,
    verbose=1
)
print(f"学習完了! 所要時間: {time.time() - start_time:.2f} 秒")

# 学習曲線の描画
plt.figure(figsize=(10, 4))
plt.plot(history.history['loss'], label='Train Loss')
plt.plot(history.history['val_loss'], label='Val Loss')
plt.title("Model Training Loss (MSE)")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()


In [ ]:
# 8. トイラジコン仕様: ステアリング「左右オンオフ（3値）」評価
# トイラジコンは連続サーボではなく左右に振り切るため、しきい値（例: ±0.25）でマッピング
STEER_THRESHOLD = 0.25

preds_s, preds_t = model.predict(X_val[:500])

def to_discrete(steer_val):
    if steer_val > STEER_THRESHOLD:
        return "RIGHT (右)"
    elif steer_val < -STEER_THRESHOLD:
        return "LEFT (左)"
    else:
        return "STRAIGHT (直進)"

actual_labels = [to_discrete(val) for val in y_s_val[:500]]
predicted_labels = [to_discrete(val[0]) for val in preds_s]

# 正解率の計算
matches = sum([1 for a, p in zip(actual_labels, predicted_labels) if a == p])
accuracy = matches / len(actual_labels) * 100.0
print(f"ステアリングオンオフ（3値）判定 正解率: {accuracy:.2f}%")

# サンプル比較の表示
print("\n[推論テストサンプル（先頭10件）]")
for i in range(10):
    print(f"#{i+1}: 正解={actual_labels[i]} (raw: {y_s_val[i]:+.2f}) -> 予測={predicted_labels[i]} (raw: {preds_s[i][0]:+.2f})")


In [ ]:
# 9. TensorFlow Lite (TFLite) への量子化エクスポート
os.makedirs("models", exist_ok=True)

# 1. 通常のFP32モデル
converter_fp32 = tf.lite.TFLiteConverter.from_keras_model(model)
tflite_fp32 = converter_fp32.convert()
with open("models/model_e2e_fp32.tflite", "wb") as f:
    f.write(tflite_fp32)

# 2. FP16量子化モデル（モバイルGPU / DSPで爆速・高精度）
converter_fp16 = tf.lite.TFLiteConverter.from_keras_model(model)
converter_fp16.optimizations = [tf.lite.Optimize.DEFAULT]
converter_fp16.target_spec.supported_types = [tf.float16]
tflite_fp16 = converter_fp16.convert()
with open("models/model_e2e_fp16.tflite", "wb") as f:
    f.write(tflite_fp16)

# 3. INT8完全整数量子化モデル（モバイルNPU / CPU向け超軽量）
def representative_data_gen():
    for i in range(100):
        yield [X_val[i:i+1]]

converter_int8 = tf.lite.TFLiteConverter.from_keras_model(model)
converter_int8.optimizations = [tf.lite.Optimize.DEFAULT]
converter_int8.representative_dataset = representative_data_gen
converter_int8.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
converter_int8.inference_input_type = tf.int8
converter_int8.inference_output_type = tf.int8
try:
    tflite_int8 = converter_int8.convert()
    with open("models/model_e2e_int8.tflite", "wb") as f:
        f.write(tflite_int8)
    int8_success = True
except Exception as e:
    print(f"INT8変換警告: {e}")
    int8_success = False

# ファイルサイズの比較
print("=== TFLite エクスポート結果 ===")
fp32_size = os.path.getsize("models/model_e2e_fp32.tflite") / 1024
fp16_size = os.path.getsize("models/model_e2e_fp16.tflite") / 1024
print(f"1. FP32 TFLite モデル: {fp32_size:.1f} KB")
print(f"2. FP16 量子化モデル: {fp16_size:.1f} KB (サイズ半減・推奨)")
if int8_success:
    int8_size = os.path.getsize("models/model_e2e_int8.tflite") / 1024
    print(f"3. INT8 量子化モデル: {int8_size:.1f} KB")


In [ ]:
# 10. スマホ実機シミュレーション: 推論レイテンシ（ms）とFPS計測
interpreter = tf.lite.Interpreter(model_path="models/model_e2e_fp16.tflite")
interpreter.allocate_tensors()

input_details = interpreter.get_input_details()
output_details = interpreter.get_output_details()

print(f"TFLite Input Shape: {input_details[0]['shape']}, Type: {input_details[0]['dtype']}")

# 100回の推論速度ベンチマーク
test_sample = X_val[0:1]
latencies = []

# ウォームアップ
for _ in range(10):
    interpreter.set_tensor(input_details[0]['index'], test_sample)
    interpreter.invoke()

for _ in range(100):
    t0 = time.perf_counter()
    interpreter.set_tensor(input_details[0]['index'], test_sample)
    interpreter.invoke()
    latencies.append((time.perf_counter() - t0) * 1000.0)

mean_latency = np.mean(latencies)
p95_latency = np.percentile(latencies, 95)
fps = 1000.0 / mean_latency

print(f"=== 推論ベンチマーク結果 (FP16モデル) ===")
print(f"平均推論時間: {mean_latency:.2f} ms")
print(f"95%タイル推論時間: {p95_latency:.2f} ms")
print(f"理論推論FPS: {fps:.1f} FPS")
print(f"判定: Galaxy Note 10+（Snapdragon 855 / Exynos 9825）なら 60FPS 以上のリアルタイム自律操舵が可能!")


In [ ]:
# モデルのダウンロード (Colab環境なら自動ダウンロード、ローカルなら保存のみ)
try:
    from google.colab import files
    print('生成されたモデルファイルをダウンロードします...')
    target_file = tflite_path if 'tflite_path' in locals() else 'models/model_e2e_fp16.tflite'
    files.download(target_file)
except ImportError:
    print('ローカル環境で実行中: モデルファイルは models/ ディレクトリに正常保存されました。')
